# 10. 조건부 mouse liver TCDD 반응 예측

실제 관측한 chemical response의 별도 경로입니다. Mouse liver에서 같은 TCDD 노출을 다른 cell type으로 전이하며, 인간 HCC 약효나 새로운 약물에 대한 일반화를 검증하지 않습니다. 기본은 저장된 실행 결과 읽기입니다.


In [1]:
from pathlib import Path
import json, subprocess
import pandas as pd
ROOT = Path.cwd()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
REPORT = ROOT / 'reports/research_phase2'


In [2]:
RUN_ID = 'C00_scvidr_mouse_TCDD_portal_seed42_v3'
run=json.loads((REPORT/'runs'/RUN_ID/'run.json').read_text())
contract=json.loads((REPORT/'runs'/RUN_ID/'data_contract.json').read_text())
assert run['status']=='executed'
print('actual run:',run['run_id'])
print('source SHA256:',run['data_sha256'])
print('training cells:',run['n_training_cells'],'heldout treated:',run['n_evaluation_cells'])
print('prediction rows:',run['n_native_prediction_rows'],'unique input controls:',run['n_unique_native_input_controls'])
print('exposure:',contract['exposure'])
print('replicate limitation:',contract['replicate'])


actual run: C00_scvidr_mouse_TCDD_portal_seed42_v3
source SHA256: f80a297d9d2ec1ad2d88886f98ffde99f6b0cc66c03e117933ed0d2662a923ea
training cells: 17791 heldout treated: 1074
prediction rows: 2344 unique input controls: 2026
exposure: Oral gavage every4days for28days, from Nault study Methods; not an acute single administration
replicate limitation: Publication reports biological replicates, but per-mouse identity is not retained as an explicit field in the supplied h5ad; batch0/1 equals treatment, not replicate


## Holdout과 대조군

전체 portal hepatocyte treated 세포를 VAE 학습과 HVG 선택에서 제외했습니다. Portal control은 입력으로 제공합니다. 5,000 HVGs는 허용된 훈련 세포에서만 고릅니다. 동일한 학습 VAE에서 native arithmetic와 native scVIDR regression을 비교하며, random seed를 맞춰 같은 native control sample을 사용했는지 실제 ID로 확인했습니다. 별도로 무변화와 다른 cell type 평균반응 기준선을 비교합니다.


In [3]:
display(pd.read_csv(REPORT/'runs'/RUN_ID/'metrics.tsv',sep='	'))


,model_id,delta_rmse,delta_mae,pearson_delta,pearson_expression,expression_r2
0,native_same_VAE_arithmetic,0.138536,0.065261,0.779404,0.893405,0.772774
1,scVIDR_regression,0.076286,0.032308,0.932575,0.965862,0.931099
2,no_change_all_target_controls,0.205685,0.069732,NaN,0.757616,0.499119
3,no_change_same_native_control_sample,0.205625,0.069741,0.033606,0.757126,0.499412
4,training_celltype_mean_response,0.130527,0.058158,0.769702,0.895505,0.798287


## 실행 변경·실패·표본 단위

scvi0.14에서는 native VIDR가 abstract setup_anndata API 때문에 생성되지 않았습니다. Parent 환경을 보존한 child 환경에 공식 코드의 scvi0.13 API를 설치했습니다. 이어 원 코드의 누락된 LinearRegression import를 보완하고, 이미 학습·저장된 v2 모델을 v3에서 복원해 예측했습니다. 실패 파일도 보존합니다. v3의 training time0은 새 학습이 없다는 뜻이며 원 학습이 무료였다는 뜻이 아닙니다.

Native control balancing은 같은 nucleus를 복수 row로 재표집할 수 있습니다. 출력 row 수를 독립 세포·동물 수로 취급하지 않습니다. 제공 파일에서 per-mouse assignment가 명시적으로 보존되지 않아 biological replicate CI를 만들지 않았습니다. Dose30μg/kg를28일 동안4일마다 투여한 조건입니다.


In [4]:
print('recovered training source:',run['recovered_training_run'])
print('time scope:',run['training_time_scope'])
print('environment:',run['environment'])
for failed in ['C00_scvidr_mouse_TCDD_portal_seed42_v1','C00_scvidr_mouse_TCDD_portal_seed42_v2']:
    failure=json.loads((REPORT/'runs'/failed/'failure.json').read_text())
    print(failed, failure['error'])


recovered training source: C00_scvidr_mouse_TCDD_portal_seed42_v2
time scope: Current run only; recovered source failure has combined preprocessing/training/failed prediction time
environment: {'scvi_tools': '0.13.0', 'torch': '1.13.1+cu117'}
C00_scvidr_mouse_TCDD_portal_seed42_v1 TypeError: Can't instantiate abstract class VIDR with abstract method setup_anndata
C00_scvidr_mouse_TCDD_portal_seed42_v2 NameError: name 'LinearRegression' is not defined


## 선택: 처음부터 재학습

실제 공식 subset과 별도 child 환경이 이미 준비된 서버에서 실행합니다. 새 run ID를 만들고 fresh를 전달하면 저장 모델 복원 없이 같은 사전 고정 설정으로 학습합니다.


In [5]:
SUBMIT_FRESH_TRAINING = False
if SUBMIT_FRESH_TRAINING:
    from datetime import datetime, timezone
    run_id='tutorial10_scvidr_'+datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S')
    result=subprocess.run(['sbatch','--parsable',str(ROOT/'slurm/scvidr_single.sh'),run_id,'fresh'],cwd=ROOT,check=True,text=True,capture_output=True)
    print('submitted:',result.stdout.strip(),'run:',run_id)
else:
    print('저장 결과 읽기 모드: 새 학습을 제출하지 않았습니다.')


저장 결과 읽기 모드: 새 학습을 제출하지 않았습니다.


데이터·원 코드: [scVIDR](https://github.com/BhattacharyaLab/scVIDR), [Nault single-dose study](https://pmc.ncbi.nlm.nih.gov/articles/PMC7674514/), [GSE148339](https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE148339). 공개 자산의 정확한 SHA와 선택 ZIP entry는 `manifests/research_phase2/scvidr_single_dose.json`에 있습니다.
